# Explore ONSV Raw Excel

Exploration of `data/raw/PERU_SINIESTROS_2008-2025.xlsx` used to fill `SHEET_MAP` in `scripts/prepare_data.py`.

## Findings summary

All sheets are **pre-aggregated pivot tables**: region rows × (year-block × category) columns, with 3–5 title/header rows on top. There is **no record-level data** (no per-accident rows, no hour 0–23 granularity).

| Sheet | Layout | Notes |
|-------|--------|-------|
| SINIESTROS AÑO REGIÓN | region × year | years on row 3, regions from row 4 |
| SINIESTROS POR TIPO | region × (year × tipo) | categories on row 4; COLISION/COLISIÓN variants |
| CAUSAS POR REGIÓN | region × (year × causa) | category set grows from 2015/2018 |
| FRANJA HORARIA | region × (year × franja) | **4 bands 2008–2014, 12 bands 2015–2024** |
| DÍA | region × (year × day) | MIÉRCOLES/SÁBADO with accents |
| FALLECIDOS / HERIDOS | region × (year × gender × age) | age scheme: 2 groups 2008–09, 6 groups 2010–17, different 6 groups 2018–24 |
| CONDUC GENERO EDAD Y LICENCIA | region × (year × mixed cols) | license columns extracted; categories vary by era |
| VEHICULOS POR TIPO Y REGIÓN | region × (year × vehicle type) | categories on row 5; renamed across years (AUTO→AUTOMOVIL) |

Data quality notes: trailing spaces in region names (`LIMA `), accent inconsistencies (`AMAZÓNAS` vs `AMAZONAS`), `TOTAL` rows at the bottom of each sheet must be excluded.

In [ ]:
import pandas as pd

RAW_EXCEL = "../data/raw/PERU_SINIESTROS_2008-2025.xlsx"
xl = pd.ExcelFile(RAW_EXCEL)
xl.sheet_names

In [ ]:
# Preview the header structure and first rows of each relevant sheet
RELEVANT = [
    "SINIESTROS AÑO REGIÓN", "SINIESTROS POR TIPO", "CAUSAS POR REGIÓN",
    "FRANJA HORARIA", "DÍA", "FALLECIDOS", "HERIDOS",
    "CONDUC GENERO EDAD Y LICENCIA", "VEHICULOS POR TIPO Y REGIÓN",
]
for sheet in RELEVANT:
    df = pd.read_excel(RAW_EXCEL, sheet_name=sheet, header=None, nrows=8)
    print("=" * 30, sheet, "=" * 30)
    display(df.iloc[:8, :12])

In [ ]:
# Map which categories exist in which years (header row 3 = years, row 4 or 5 = categories)
from collections import defaultdict

def scan_categories(sheet, cat_row):
    df = pd.read_excel(RAW_EXCEL, sheet_name=sheet, header=None)
    years, cats = df.iloc[3].ffill(), df.iloc[cat_row]
    by_year = defaultdict(set)
    for c in range(1, df.shape[1]):
        if pd.notna(years[c]) and pd.notna(cats[c]):
            by_year[int(years[c])].add(str(cats[c]).strip())
    for cat in sorted(set().union(*by_year.values())):
        yrs = [y for y in by_year if cat in by_year[y]]
        print(f"  {cat!r}: {min(yrs)}-{max(yrs)}")

scan_categories("FRANJA HORARIA", 4)